# Custom Store
> Implement the storage protocols with your own backends.

Anchor defines three storage protocols: `ContextStore`, `VectorStore`, and
`MemoryEntryStore`.  Any class that implements the required methods can be
used as a drop-in replacement — no inheritance needed.

**Time:** ~7 minutes

## Setup

In [ ]:
from anchor.protocols import ContextStore, VectorStore, MemoryEntryStore

## Protocol Overview
Each protocol requires a small set of methods:

| Protocol | Methods |
|---|---|
| `ContextStore` | `add()`, `get()`, `get_all()` |
| `VectorStore` | `add_embedding()`, `search()`, `delete()`, `clear()` |
| `MemoryEntryStore` | `add()`, `update()`, `delete()`, `list_all()` |

## Implement a Custom ContextStore
A minimal dictionary-backed context store.

In [ ]:
class DictContextStore:
    """Simple dict-backed ContextStore."""

    def __init__(self):
        self._items = {}

    def add(self, item):
        self._items[item.id] = item
        print(f"  Added context item '{item.id}'")

    def get(self, item_id: str):
        return self._items.get(item_id)

    def get_all(self):
        return list(self._items.values())


ctx_store = DictContextStore()
print(f"Satisfies ContextStore: {isinstance(ctx_store, ContextStore)}")

## Implement a Custom VectorStore
A stub that stores vectors in a list.

In [ ]:
class ListVectorStore:
    """List-backed VectorStore for demonstration."""

    def __init__(self):
        self._vectors = []

    def add_embedding(self, id: str, embedding: list, metadata: dict = None):
        self._vectors.append({"id": id, "embedding": embedding, "metadata": metadata or {}})
        print(f"  Added embedding '{id}' (dim={len(embedding)})")

    def search(self, query_embedding: list, top_k: int = 5) -> list:
        # Simplified: return all entries (real impl would compute similarity)
        print(f"  Searching top-{top_k} among {len(self._vectors)} vectors")
        return self._vectors[:top_k]

    def delete(self, id: str):
        self._vectors = [v for v in self._vectors if v["id"] != id]
        print(f"  Deleted '{id}'")

    def clear(self):
        self._vectors.clear()
        print("  Cleared all vectors")


vec_store = ListVectorStore()
print(f"Satisfies VectorStore: {isinstance(vec_store, VectorStore)}")

## Implement a Custom MemoryEntryStore
A dictionary-backed store for memory entries.

In [ ]:
class DictEntryStore:
    """Dict-backed MemoryEntryStore."""

    def __init__(self):
        self._entries = {}

    def add(self, entry):
        self._entries[entry.id] = entry
        print(f"  Added entry '{entry.id}'")

    def update(self, entry):
        self._entries[entry.id] = entry
        print(f"  Updated entry '{entry.id}'")

    def delete(self, entry_id: str):
        self._entries.pop(entry_id, None)
        print(f"  Deleted entry '{entry_id}'")

    def list_all(self):
        return list(self._entries.values())


entry_store = DictEntryStore()
print(f"Satisfies MemoryEntryStore: {isinstance(entry_store, MemoryEntryStore)}")

## Exercise All Three Stores
Quick round-trip to confirm each store works.

In [ ]:
from anchor.models import ContextItem, SourceType, MemoryEntry

# ContextStore
print("=== ContextStore ===")
ctx_item = ContextItem(
    id="c1", content="test context",
    source=SourceType.RETRIEVAL, score=0.9, priority=5, token_count=2,
)
ctx_store.add(ctx_item)
print(f"  get('c1'): {ctx_store.get('c1').content}")
print(f"  get_all(): {len(ctx_store.get_all())} items")

# VectorStore
print("\n=== VectorStore ===")
vec_store.add_embedding(id="v1", embedding=[0.1, 0.2], metadata={"tag": "demo"})
results = vec_store.search(query_embedding=[0.1, 0.2], top_k=3)
print(f"  Results: {len(results)}")
vec_store.delete("v1")
vec_store.clear()

# MemoryEntryStore
print("\n=== MemoryEntryStore ===")
mem = MemoryEntry(
    id="m1", content="test memory",
    relevance_score=0.7, metadata={},
)
entry_store.add(mem)
mem.content = "updated memory"
entry_store.update(mem)
print(f"  list_all(): {len(entry_store.list_all())} entries")
entry_store.delete("m1")
print(f"  After delete: {len(entry_store.list_all())} entries")

## Key Takeaways
- Three storage protocols: `ContextStore`, `VectorStore`, `MemoryEntryStore`
- All are structural (duck-typed) — implement the methods, skip the base class
- `isinstance()` confirms conformance at runtime
- Custom backends can wrap databases, cloud APIs, or any persistence layer

**Previous:** [JSON File Store](05_json_file_store.ipynb)